In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import csv
from pathlib import Path

# Загружаем модель и изображения
model = YOLO('D:/Data_env/Inf_cell_v26L-seg.pt')    # модель
input_dir = Path('D:/Data_env/cells')    # папка с изображениями
output_csv = Path('D:/Data_env/cells/results.csv')    # файл, куда запишется результат

# Расширения файлов изображений
image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.tif')

# Поиск всех изображений для анализа
image_files = []
for ext in image_extensions:
    image_files.extend(input_dir.glob(ext))
print(f'Найдено изображений для обработки: {len(image_files)}')

# Запрашиваем текст для колонки 'File Name'
print("="*30)
custom_name = input("Введите имя для колонки 'File Name', которое будет записано для всех строк: ").strip()
print("="*30 + "\n")

# Создаем файл для записи результатов
with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['File Name', 
                     'Total Pixels', 
                     'Mask Pixels', 
                     'Coverage Percentage', 
                     'Objects Detected'])

    # Основной цикл обработки
    for img_path in image_files:
        try:
            img = cv2.imread(str(img_path))
            if img is None:
                print(f'Ошибка загрузки файла (пропущен): {img_path.name}')
                continue

            h, w, _ = img.shape
            total_pixels = h * w

            # Запуск сегментации
            results = model(img_path, verbose=False, conf=0.1)     # порог подбирается эмпирически

            # Создание пустой маски
            all_masks = np.zeros((h, w), dtype=np.uint8)
            object_count = 0

            for result in results:
                if result.masks is not None:
                    # Подсчет объектов
                    object_count += len(result.masks.xy)
                    
                    # Отрисовка полигонов на маске
                    for segments in result.masks.xy:
                        points = np.array(segments, dtype=np.int32)
                        cv2.fillPoly(all_masks, [points], 255)

            # Расчет пикселей и покрытия
            mask_pixels = int(np.sum(all_masks == 255))
            coverage_pct = (mask_pixels / total_pixels) * 100 if total_pixels > 0 else 0

            # Если ввода нет, то оставляем оригинальное имя файла img_path.name
            record_name = custom_name if custom_name else img_path.name

            # Запись
            writer.writerow([record_name, total_pixels, mask_pixels, round(coverage_pct, 4), object_count])
            print(f'✅ Обработан {img_path.name} | Маски: {object_count} | Покрытие: {coverage_pct:.2f}%')

        except Exception as e:
            print(f'Ошибка при обработке {img_path.name}: {e}')

# --- БЛОК ПЕРЕИМЕНОВАНИЯ ---
print("\n" + "="*30)
user_filename = input("Введите желаемое имя для CSV-файла (без расширения .csv): ").strip()

if user_filename:
    # Формируем новый путь
    final_csv = input_dir / f"{user_filename}.csv"
    
    # Переименовываем файл на диске
    output_csv.rename(final_csv)
    print(f"Обработка завершена. Результаты успешно сохранены в файл: {final_csv.name}")
else:
    print(f"Обработка завершена. Имя не введено. Файл сохранен как: {output_csv.name}")

Найдено изображений для обработки: 14


Введите имя для колонки 'File Name', которое будет записано для всех строк:  Inf_3115



✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_10.jpg | Маски: 12 | Покрытие: 54.09%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_4.jpg | Маски: 17 | Покрытие: 51.07%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_5.jpg | Маски: 19 | Покрытие: 64.35%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_6.jpg | Маски: 11 | Покрытие: 59.96%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_7.jpg | Маски: 15 | Покрытие: 70.86%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_8.jpg | Маски: 17 | Покрытие: 72.19%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_9.jpg | Маски: 16 | Покрытие: 67.93%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_10.jpg | Маски: 8 | Покрытие: 80.43%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_4.jpg | Маски: 14 | Покрытие: 70.02%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_5.jpg | Маски: 13 | Покрытие: 66.52%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_6.jpg | Маски: 12 | Покрытие: 64.80%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_7.jpg | Маски: 3 | Покрытие

Введите желаемое имя для CSV-файла (без расширения .csv):  Inf_by_3115


Обработка завершена. Результаты успешно сохранены в файл: Inf_by_3115.csv


---